# Metacritic Game Site Web Scrape - Full Pipeline Write-Up

After exploring the structures of Metacritic's pages and where to find the right classes for the information we seek, We'll construct the full pipeline here.  

**Broad Work Flow**:
1. Prepare information for later lookup
    - game title
    - platforms
    - all necessary urls (use `data/mappings/platform_slugs.json`)
2. Identify which game x platform row has missing values in critic as well as user data
3. Conduct scrapes separately for
    - critic scores and review counts
    - user scores and review counts
    - game stats from backend urls
4. Aggregate all new scraped data into one df
5. Merge acquired data sensibly with original data set
6. Profit!

## Setup

In [2]:
# +++ Import all necessary modules +++

import time
from urllib.parse import parse_qs, urlparse

import pandas as pd
import requests
from bs4 import BeautifulSoup

from core.config import DATA_FORMATTED_PATH

In [4]:
# +++ GLOBAL SETTINGS +++

# Percentage of unique game titles to choose as batch
BATCH_PCT = 0.005


## Load data and define data sample to work with

In [5]:
# load the data
df = pd.read_csv(DATA_FORMATTED_PATH)

unique_titles = df['title'].drop_duplicates()
n_games_total = df['title'].nunique()

# get batch size and corresponding game titles
batch_size = round(n_games_total * BATCH_PCT)
batch_titles = unique_titles.iloc[:batch_size]

# extract batch data
df_batch = df[df['title'].isin(batch_titles)].copy()

print(f"Games in full dataset:  {n_games_total}")
print(f"Games selected:         {len(batch_titles)}")
print(f"Game × platform rows:   {len(df_batch)}")

Games in full dataset:  13429
Games selected:         67
Game × platform rows:   178


## 1. Prepare Look-Up Information